# Test your fine-tuned T5 question-answering model

Attach notebook 2's saved outputs as a Kaggle input, including the selected
`checkpoints/t5_small/seed_42/hf_export/` folder. The earlier successful run's
export works too; retraining is not required to use this notebook.
Set `REPO_URL`, clone/install the Python helpers, and select your export below.
No dataset download, training, or Mamba installation runs here. GPU is optional.

This model answers questions from a supplied English passage. Provide the
context containing the answer; it was not trained as a general chatbot.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = ""  # fill in your GitHub repository URL after pushing the project
REPO_REF = "main"  # preferably the same commit SHA in BOTH notebooks
SOURCE_SUBDIR = "."  # edit if this assignment lives below the repository root
WORKSPACE = Path("/kaggle/working")
CLONE_ROOT = WORKSPACE / "qa_assignment_source"
OUTPUT_ROOT = WORKSPACE / "t5_qa_test"
RESTORE_FROM = None  # e.g. Path('/kaggle/input/prior-qa-output/qa_assignment')

if not REPO_URL.strip():
    raise ValueError("Set REPO_URL to the GitHub repository you pushed this project to.")
if not WORKSPACE.is_dir():
    raise RuntimeError("This notebook is intended for Kaggle. Run local tests with pytest.")
if not (CLONE_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, str(CLONE_ROOT)], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
REPO_ROOT = (CLONE_ROOT / SOURCE_SUBDIR).resolve()
assert (REPO_ROOT / "pyproject.toml").is_file(), "Check SOURCE_SUBDIR."
print("Source:", REPO_ROOT)
print("Artifacts:", OUTPUT_ROOT)
subprocess.run(["git", "-C", str(CLONE_ROOT), "rev-parse", "HEAD"], check=True)

## Install the Python helpers

Run setup before importing project modules. These requirements pin the Hugging
Face interface used here and preserve Kaggle's preinstalled CUDA-enabled PyTorch.
Setup also adds the cloned `src/` directory to this running kernel's import path;
a subprocess editable install alone cannot refresh an already-started kernel.
If you previously imported Transformers in this session, restart the session
after installing and rerun from the top. A successful import is followed by an
actual forward/backward and generation probe before training.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_ROOT / "requirements-kaggle.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "-e", str(REPO_ROOT)], check=True)

# Editable-install .pth files are read at interpreter startup. This kernel
# predates the pip subprocess, so explicitly expose the cloned sources now.
import importlib

SOURCE_ROOT = (REPO_ROOT / "src").resolve()
if not (SOURCE_ROOT / "qa_assignment" / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing qa_assignment sources under {SOURCE_ROOT}; check SOURCE_SUBDIR.")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()

import qa_assignment
if Path(qa_assignment.__file__).resolve().parent != SOURCE_ROOT / "qa_assignment":
    raise RuntimeError("qa_assignment was loaded from another checkout. Restart the session and rerun setup.")
print("Loaded helpers:", qa_assignment.__file__)

import torch
from qa_assignment.utils import environment_info
from qa_assignment.runtime import restore_artifacts

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Inference device:", DEVICE)


## Load your saved model

`MODEL_DIR` is a local folder, for example
`/kaggle/input/YOUR-SAVED-OUTPUT/qa_baselines/checkpoints/t5_small/seed_42/hf_export`.
Enter the exact path shown below. If only one T5 export is attached, an empty
setting selects it automatically. With multiple exports, choose one explicitly.
The model and tokenizer load entirely from the export, without fetching new weights.
Run this cell once; subsequent question cells reuse the loaded model.

In [ ]:
from qa_assignment.inference import T5Answerer, find_t5_exports

MODEL_DIR = ""  # path to the attached hf_export/ folder; not best.pt
exports = find_t5_exports([Path("/kaggle/input"), Path("/kaggle/working")])
print("Available T5 exports:")
for index, path in enumerate(exports):
    print(index, path)
if not MODEL_DIR.strip():
    if len(exports) != 1:
        raise ValueError("Attach your saved output, then set MODEL_DIR to one listed hf_export folder.")
    MODEL_DIR = str(exports[0])
qa = T5Answerer.from_export(MODEL_DIR, device=DEVICE)
print("Loaded:", MODEL_DIR)
print("Question + passage limit:", qa.max_input_length, "tokens")

## Your passage and question

Replace the passage and question below, then rerun these cells to ask more.
The complete question plus passage must fit the saved input limit (normally
512 tokens); overly long inputs are rejected so the answer's context is not
silently discarded. Questions with no answer in the passage can still receive
an incorrect answer: this SQuAD 1.1 model has no trained abstention behavior.

In [ ]:
PASSAGE = (
    "Super Bowl 50 was played on February 7, 2016, at Levi's Stadium in "
    "Santa Clara, California. The Denver Broncos defeated the Carolina "
    "Panthers by a score of 24-10. Von Miller was named the game's MVP."
)
QUESTION = "Which team won Super Bowl 50?"
print("Answer:", qa.answer(QUESTION, PASSAGE) or "<empty answer>")

## Ask several questions about the same passage

Edit this list and rerun the cell. The model stays loaded. Each question is
answered independently from the passage; prior answers are not chat history.

In [ ]:
QUESTIONS = [
    "Where was Super Bowl 50 played?",
    "Who was named the game's MVP?",
    "When was Super Bowl 50 played?",
]
for result in qa.answer_many(QUESTIONS, PASSAGE):
    print("Question:", result["question"])
    print("Answer:", result["answer"] or "<empty answer>")

## Optional typed questions and saved answers

For a manual session, enable the input loop below. Submit an empty question
to exit. It defaults off so a saved Kaggle run cannot wait indefinitely for input.
Answers from either interface can be saved in Kaggle Output.

In [ ]:
RUN_INTERACTIVE_CHAT = False
if RUN_INTERACTIVE_CHAT:
    qa.interactive(PASSAGE)

In [ ]:
from qa_assignment.utils import write_json

answer_log = OUTPUT_ROOT / "answers.json"
write_json(answer_log, {"model_dir": MODEL_DIR, "answers": qa.history})
print("Saved answers:", answer_log)